# ABP Fieldization Sanity Check

This notebook checks a continuous WCA-ABP simulation and the particle-size-aware mapping from particle centers to Eulerian fields.  The default field is a center count field, not a clipped 0/1 hard-core occupancy.  A Gaussian cloud option is also shown for comparison.

In [ ]:
import os
import sys

candidate_roots = [
    os.environ.get("CNEEP_V2_ROOT"),
    os.path.abspath("../.."),
    os.path.abspath(".."),
    os.path.abspath("."),
    "/home/user1/CNEEP_v2",
]

CNEEP_V2_ROOT = None
for candidate in candidate_roots:
    if candidate and os.path.exists(os.path.join(candidate, "data", "ABP", "core.py")):
        CNEEP_V2_ROOT = candidate
        break

if CNEEP_V2_ROOT is None:
    raise RuntimeError("Could not locate CNEEP_v2 root. Set CNEEP_V2_ROOT.")

if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)

print("CNEEP_v2 root:", CNEEP_V2_ROOT)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from data.ABP import ABPParams, ContinuousABP, ABPFieldizer, recommended_center_grid_size

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Simulate a small WCA-ABP ensemble

In [ ]:
params = ABPParams(
    N=196,
    L=20.0,
    sigma=1.0,
    epsilon=1.0,
    mobility=1.0,
    force_clip=500.0,
    force_chunk_size=256,
    v0=8.0,
    Dr=1.0,
    Dt=0.02,
    dt=2.0e-4,
    seed=7,
    device=device,
)

grid_center = max(32, recommended_center_grid_size(params.L, params.sigma))
fieldizer = ABPFieldizer(
    box_size=params.L,
    grid_size=grid_center,
    particle_diameter=params.sigma,
    mode="center",
    include_orientation=True,
    clip_occupancy=False,
)

print(f"packing fraction phi={params.phi:.3f}, Pe={params.Pe:.2f}")
print(f"grid={grid_center}x{grid_center}, dx={fieldizer.dx:.4f}, sigma/sqrt(2)={params.sigma / math.sqrt(2):.4f}")

sim = ContinuousABP(params)
result = sim.simulate(
    B=2,
    burn_in=200,
    n_steps=600,
    save_interval=30,
    fieldizer=fieldizer,
    show_progress=True,
)

positions = result["positions"]
theta = result["theta"]
fields = result["fields"]
print("positions:", positions.shape)
print("theta:    ", theta.shape)
print("fields:   ", fields.shape)

## 2. Center-count diagnostics

In [ ]:
last_pos = positions[-1].to(device)
diag = fieldizer.diagnostics_dict(last_pos)
for k, v in diag.items():
    print(f"{k}: {v}")

print(f"min pair distance / sigma: {(result['min_distance'][-1].min() / params.sigma).item():.3f}")
print("Count fields keep multi-center pixels instead of clipping them to 1.")

## 3. Particle state and center field

In [ ]:
frame = -1
ens = 0
pos = positions[frame, ens].numpy()
ang = theta[frame, ens].numpy()
field = fields[frame, ens].numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(pos[:, 0], pos[:, 1], s=8, alpha=0.8)
stride = max(1, params.N // 60)
axes[0].quiver(
    pos[::stride, 0], pos[::stride, 1],
    np.cos(ang[::stride]), np.sin(ang[::stride]),
    angles="xy", scale_units="xy", scale=4, width=0.004,
)
axes[0].set_xlim(0, params.L)
axes[0].set_ylim(0, params.L)
axes[0].set_aspect("equal")
axes[0].set_title("particles")

im1 = axes[1].imshow(field[0].T, origin="lower", cmap="viridis")
axes[1].set_title("center count")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

orient_mag = np.sqrt(field[1] ** 2 + field[2] ** 2)
im2 = axes[2].imshow(orient_mag.T, origin="lower", cmap="viridis", vmin=0, vmax=1)
axes[2].set_title("orientation-channel magnitude")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

## 4. Center count vs finite-radius and Gaussian fields

In [ ]:
center_fieldizer = ABPFieldizer(
    params.L, grid_center, params.sigma,
    mode="center", include_orientation=False, clip_occupancy=False,
)
disk_fieldizer = ABPFieldizer(
    params.L, grid_center, params.sigma,
    mode="disk", include_orientation=False, clip_occupancy=False,
)
gaussian_fieldizer = ABPFieldizer(
    params.L, grid_center, params.sigma,
    mode="gaussian", include_orientation=False, clip_occupancy=False,
    gaussian_sigma=0.5 * params.sigma,
)

center_field = center_fieldizer.encode(last_pos[:1])[0, 0].cpu().numpy()
disk_field = disk_fieldizer.encode(last_pos[:1])[0, 0].cpu().numpy()
gaussian_field = gaussian_fieldizer.encode(last_pos[:1])[0, 0].cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
im0 = axes[0].imshow(center_field.T, origin="lower", cmap="viridis")
axes[0].set_title("center count")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(disk_field.T, origin="lower", cmap="magma")
axes[1].set_title("disk footprint count")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(gaussian_field.T, origin="lower", cmap="viridis")
axes[2].set_title("Gaussian cloud")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

print(f"center count sum:   {center_field.sum():.6f}")
print(f"disk footprint sum: {disk_field.sum():.6f}")
print(f"gaussian sum:       {gaussian_field.sum():.6f}")
print(f"N particles:        {params.N}")

## 5. Basic WCA stability checks

In [ ]:
time = result["times"].numpy()
potential = result["potential"].numpy()
min_distance = result["min_distance"].numpy()
mean_force = result["mean_force_norm"].numpy()

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(time, potential)
axes[0].set_ylabel("WCA potential")
axes[0].set_title("Potential should stay finite")

axes[1].plot(time, min_distance / params.sigma)
axes[1].axhline(1.0, color="k", linestyle="--", lw=1, label="sigma")
axes[1].set_ylabel("min distance / sigma")
axes[1].legend()

axes[2].plot(time, mean_force)
axes[2].set_ylabel("mean |F|")
axes[2].set_xlabel("time")

plt.tight_layout()
plt.show()